# Coding Masked Self-Attention in PyTorch

* Self-Attention plays a key role in Encoder only models like embedding models (like BERT).
* Masked Self-Attention plays a key role in Decoder only models like Generative Models (like LLMs (like ChatGPT)).
* The simple difference between Self-Attention and Masked Self-Attention is...
    * Self-Attention considers all the tokens before and after including it while calculating the attention scores.
    * Masked Self-Attention considers all the tokens before it including it and ignores any token that comes after it for calculating attention scores. This will help it predict the next word and that is the basis of the Generative AI.

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
class MaskedSelfAttention(nn.Module):
    def __init__(self, d_model=2, row_dim=0, col_dim=1):
        super().__init__()
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.row_dim = row_dim
        self.col_dim = col_dim
        
    def forward(self, token_encodings, mask=None):
        q = self.W_q(token_encodings)
        k = self.W_k(token_encodings)
        v = self.W_v(token_encodings)
        
        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))
        scaled_sims = sims/torch.tensor(k.size(self.col_dim)**0.5)
        if mask is not None:
            scaled_sims = scaled_sims.masked_fill(mask=mask, value=-1e9)
        
        attention_proportions = F.softmax(scaled_sims, dim=self.col_dim)
        attention_scores = torch.matmul(attention_proportions, v)
        return attention_scores

In [3]:
# Calculating the masked attention scores
encoding_matrix = torch.tensor([[1.16, 0.23],
                                [0.57, 1.36],
                                [4.41, -2.16]])

torch.manual_seed(42)

masked_self_attention = MaskedSelfAttention(d_model=2, row_dim=0, col_dim=1)

In [ ]:
mask = torch.tril(torch.ones(3, 3)) # tri-triangle; l-lower -> tril; for upper triangle it will be triu
mask

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [ ]:
mask = mask == 0 # masked fill only supports boolean masks # Trues corresponds to the value that we want to mask out.

In [11]:
mask

tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])

In [12]:
masked_self_attention(encoding_matrix, mask)

tensor([[ 0.6038,  0.7434],
        [-0.0062,  0.6072],
        [ 3.4989,  2.2427]], grad_fn=<MmBackward0>)

In [16]:
# Printing out the weights # Weights when multiplied to the input values, they will be transposed, so transposing here

masked_self_attention.W_q.weight.transpose(0, 1)

tensor([[ 0.5406, -0.1657],
        [ 0.5869,  0.6496]], grad_fn=<TransposeBackward0>)

In [18]:
masked_self_attention.W_k.weight.transpose(0, 1)

tensor([[-0.1549, -0.3443],
        [ 0.1427,  0.4153]], grad_fn=<TransposeBackward0>)

In [19]:
masked_self_attention.W_v.weight.transpose(0, 1)

tensor([[ 0.6233,  0.6146],
        [-0.5188,  0.1323]], grad_fn=<TransposeBackward0>)

In [23]:
# Calculate the Query Matrix

q = masked_self_attention.W_q(encoding_matrix)
q

tensor([[ 0.7621, -0.0428],
        [ 1.1063,  0.7890],
        [ 1.1164, -2.1336]], grad_fn=<MmBackward0>)

In [24]:
k = masked_self_attention.W_k(encoding_matrix)
k

tensor([[-0.1469, -0.3038],
        [ 0.1057,  0.3685],
        [-0.9914, -2.4152]], grad_fn=<MmBackward0>)

In [25]:
v = masked_self_attention.W_v(encoding_matrix)
v

tensor([[ 0.6038,  0.7434],
        [-0.3502,  0.5303],
        [ 3.8695,  2.4246]], grad_fn=<MmBackward0>)

In [28]:
sims = torch.matmul(q, k.transpose(dim0=0, dim1=1))

In [40]:
scaled_sims = sims/torch.tensor(k.size(1)**0.5)

In [41]:
scaled_sims

tensor([[-0.0700,  0.0458, -0.4612],
        [-0.2844,  0.2883, -2.1230],
        [ 0.3424, -0.4725,  2.8610]], grad_fn=<DivBackward0>)

In [42]:
scaled_sims = scaled_sims.masked_fill(mask=mask, value=-1e9)
scaled_sims

tensor([[-6.9975e-02, -1.0000e+09, -1.0000e+09],
        [-2.8442e-01,  2.8833e-01, -1.0000e+09],
        [ 3.4241e-01, -4.7253e-01,  2.8610e+00]],
       grad_fn=<MaskedFillBackward0>)

In [45]:
attention_proportions = F.softmax(scaled_sims, dim=1)
attention_proportions

tensor([[1.0000, 0.0000, 0.0000],
        [0.3606, 0.6394, 0.0000],
        [0.0722, 0.0320, 0.8959]], grad_fn=<SoftmaxBackward0>)

In [47]:
attention_scores = torch.matmul(attention_proportions, v)
attention_scores

tensor([[ 0.6038,  0.7434],
        [-0.0062,  0.6072],
        [ 3.4989,  2.2427]], grad_fn=<MmBackward0>)